# Step 5E.3 — Targeted ablation (Phase 3): incremental внесок cve_ctx_service vs решти CTI

Виконується ПІСЛЯ Step 5E.1 (overlap audit) і Step 5E.2 (feature attribution, reproducibility
audit пройшов для всіх 4 frozen моделей), який показав різке розходження в model attribution:
CTI забирає 81.98% сукупного gain у CIC-F1 (з них 81.94% -- `cve_ctx_service`), але лише
0.49% у UNSW-F1 (0.43% -- `cve_ctx_service`). Це узгоджується з напрямком ΔAP (CIC +0.263659,
UNSW +0.010317), але Step 5E.2 явно НЕ встановлює причинний внесок -- gain показує лише де
модель використовує сигнал, а не що спричиняє приріст AP.

**Дизайн Phase 3 (targeted ablation, per specification):** для кожного датасету навчаємо
4 моделі за тим самим frozen-протоколом (seed=42, ідентичні гіперпараметри, той самий
train/val/test split):

1. `F0` -- baseline (без CTI), контроль.
2. `F0 + cve_ctx_service` -- лише одна CTI-ознака додана до F0.
3. `F0 + решта CTI` -- усі CTI-ознаки КРІМ `cve_ctx_service` додані до F0.
4. `F1` -- повний набір CTI (контроль верхньої межі, вже підтверджений Step 5D.2/5E.2).

Обчислюємо:

$$\Delta AP_{service} = AP(F0 + service) - AP(F0)$$
$$\Delta AP_{other} = AP(F0 + \text{решта CTI}) - AP(F0)$$
$$\Delta AP_{all} = AP(F1) - AP(F0)$$

**Методологічні застереження (явно, а не мовчки):**
- Це **один-єдиний frozen прогін** на кожен набір ознак (seed=42, без bootstrap/повторних
  seed-ів) -- тому немає оцінки дисперсії чи довірчого інтервалу для жодного ΔAP. Звітуємо
  точкові оцінки і порівнюємо їх ВЕЛИЧИНИ, а не робимо статистичний тест значущості різниці.
- Дерева бустингу НЕ є лінійними адитивними моделями за ознаками: НЕМАЄ апріорної підстави
  очікувати точної рівності $\Delta AP_{service} + \Delta AP_{other} = \Delta AP_{all}$ --
  взаємодія ознак (напр. `cve_ctx_service` та `cve_ctx_pool_size`) може робити внесок
  комбінації CTI-ознак більшим або меншим за суму окремих внесків. Це перевіряється нижче як
  діагностика, а не як assert.
- Ablation з однією доданою ознакою (`F0 + service`) НЕ ізолює причинний ефект у строгому
  сенсі контрольованого експерименту (немає рандомізації, немає повторів) -- це найкраща
  наближена causal-подібна перевірка, доступна в межах single frozen-протоколу, і саме так
  вона й буде описана у статті: "incremental contribution under this single frozen training
  run", а не "causally proven contribution".

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
AUDIT_OUT_DIR = RESULTS_DIR / "step5e3_targeted_ablation"
AUDIT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 1. Frozen конфігурація і explicit ordered feature lists для 4 ablation-наборів на датасет

Буквально ті самі `FROZEN_F0_FEATURES` і `F1_ADDITIONS`, що й у Step 5D.2b/Step 5E.2 (та сама
frozen контракт-дисципліна: явний ordered список, list-equality assert, ніякої
algorithm-based реконструкції).

In [2]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_STATE = 42
LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True

FROZEN_BOUNDARIES = {
    "CIC": dict(
        file="cic_ids2017_with_cve_context.parquet", time_col="Timestamp",
        label_col="attack_label", benign_value="BENIGN",
        boundary_1=pd.Timestamp("2017-07-06 09:28:00"),
        boundary_2=pd.Timestamp("2017-07-07 03:23:00"),
    ),
    "UNSW": dict(
        file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
        label_col="attack_cat", benign_value="Normal",
        boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
        boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
    ),
}

FROZEN_F0_FEATURES = {
    "CIC": [
        "Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Total Length of Fwd Packets",
        "Total Length of Bwd Packets", "Fwd Packet Length Max", "Fwd Packet Length Min",
        "Fwd Packet Length Mean", "Fwd Packet Length Std", "Bwd Packet Length Max",
        "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
        "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
        "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max",
        "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max",
        "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
        "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
        "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
        "Packet Length Variance", "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
        "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
        "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
        "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
        "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
        "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean",
        "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max",
        "Idle Min", "protocol",
    ],
    "UNSW": [
        "state", "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "Sload", "Dload",
        "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
        "res_bdy_len", "Sjit", "Djit", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
        "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
        "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
        "ct_dst_sport_ltm", "ct_dst_src_ltm", "proto", "service",
    ],
}
F1_ADDITIONS = {
    "CIC": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_epss_coverage",
        "cve_ctx_service_max_epss", "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
    "UNSW": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_epss_coverage",
        "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
}
OTHER_CTI_ADDITIONS = {
    dataset_key: [c for c in cols if c != "cve_ctx_service"]
    for dataset_key, cols in F1_ADDITIONS.items()
}

FEATURE_SETS = {}
for dataset_key in ["CIC", "UNSW"]:
    FEATURE_SETS[(dataset_key, "F0")] = list(FROZEN_F0_FEATURES[dataset_key])
    FEATURE_SETS[(dataset_key, "F0_service")] = list(FROZEN_F0_FEATURES[dataset_key]) + ["cve_ctx_service"]
    FEATURE_SETS[(dataset_key, "F0_other_cti")] = list(FROZEN_F0_FEATURES[dataset_key]) + list(OTHER_CTI_ADDITIONS[dataset_key])
    FEATURE_SETS[(dataset_key, "F1")] = list(FROZEN_F0_FEATURES[dataset_key]) + list(F1_ADDITIONS[dataset_key])

EXPECTED_FEATURE_COUNTS = {
    ("CIC", "F0"): 77, ("CIC", "F0_service"): 78, ("CIC", "F0_other_cti"): 83, ("CIC", "F1"): 84,
    ("UNSW", "F0"): 41, ("UNSW", "F0_service"): 42, ("UNSW", "F0_other_cti"): 45, ("UNSW", "F1"): 46,
}
for key, expected in EXPECTED_FEATURE_COUNTS.items():
    n = len(FEATURE_SETS[key])
    assert n == expected, f"{key}: n={n}, очікувалось {expected}"
print("✅ FEATURE_SETS (4 ablation-набори x 2 датасети) == очікувані розміри")
for key in sorted(FEATURE_SETS, key=lambda k: (k[0], k[1])):
    print(f"  {key}: {len(FEATURE_SETS[key])} ознак")

# --- Незалежна перевірка ПОРЯДКУ ознак (не лише кількості) -----------------------------
# EXPECTED_FEATURE_COUNTS вище перевіряє лише len(...) -- цього недостатньо: саме порядок
# колонок є frozen-контрактом цього проєкту (прецедент UNSW, Step 5D.2b: однакова кількість
# ознак і Total Bins все одно дали різний Booster через порядок). Тому нижче -- ОКРЕМО
# ретипований (не виведений з FEATURE_SETS вище) літеральний reference-словник для F0-частин,
# з якого F1/service/other_cti деривуються ІНШИМ кодовим шляхом, ніж у побудові FEATURE_SETS
# вище. Це ловить помилку саме у логіці збирання FEATURE_SETS (неправильна конкатенація,
# втрачений list(...), переставлений порядок додавання), а не помилку у самому
# FROZEN_F0_FEATURES -- та узгодженість вже перевірена окремо через feature_contract.json +
# SHA-256 hash у Step 5D.2b.
FROZEN_REFERENCE_FEATURES = {}
FROZEN_REFERENCE_FEATURES[("CIC", "F0")] = [
    "Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Total Length of Fwd Packets",
    "Total Length of Bwd Packets", "Fwd Packet Length Max", "Fwd Packet Length Min",
    "Fwd Packet Length Mean", "Fwd Packet Length Std", "Bwd Packet Length Max",
    "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
    "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
    "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max",
    "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max",
    "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
    "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
    "Packet Length Variance", "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
    "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
    "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
    "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
    "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
    "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean",
    "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max",
    "Idle Min", "protocol",
]
FROZEN_REFERENCE_FEATURES[("UNSW", "F0")] = [
    "state", "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "Sload", "Dload",
    "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
    "res_bdy_len", "Sjit", "Djit", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
    "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
    "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
    "ct_dst_sport_ltm", "ct_dst_src_ltm", "proto", "service",
]
CIC_CTI_ALL = ["cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_epss_coverage",
               "cve_ctx_service_max_epss", "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches"]
UNSW_CTI_ALL = ["cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_epss_coverage",
                "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches"]
FROZEN_REFERENCE_FEATURES[("CIC", "F1")] = FROZEN_REFERENCE_FEATURES[("CIC", "F0")] + CIC_CTI_ALL
FROZEN_REFERENCE_FEATURES[("CIC", "F0_service")] = FROZEN_REFERENCE_FEATURES[("CIC", "F0")] + ["cve_ctx_service"]
FROZEN_REFERENCE_FEATURES[("CIC", "F0_other_cti")] = FROZEN_REFERENCE_FEATURES[("CIC", "F0")] + [c for c in CIC_CTI_ALL if c != "cve_ctx_service"]
FROZEN_REFERENCE_FEATURES[("UNSW", "F1")] = FROZEN_REFERENCE_FEATURES[("UNSW", "F0")] + UNSW_CTI_ALL
FROZEN_REFERENCE_FEATURES[("UNSW", "F0_service")] = FROZEN_REFERENCE_FEATURES[("UNSW", "F0")] + ["cve_ctx_service"]
FROZEN_REFERENCE_FEATURES[("UNSW", "F0_other_cti")] = FROZEN_REFERENCE_FEATURES[("UNSW", "F0")] + [c for c in UNSW_CTI_ALL if c != "cve_ctx_service"]

for key in FEATURE_SETS:
    assert FEATURE_SETS[key] == FROZEN_REFERENCE_FEATURES[key], f"{key}: ПОРЯДОК ознак розійшовся з незалежним frozen reference!"
print("✅ Порядок ознак (не лише кількість) у всіх 8 FEATURE_SETS збігається з незалежним FROZEN_REFERENCE_FEATURES")


✅ FEATURE_SETS (4 ablation-набори x 2 датасети) == очікувані розміри
  ('CIC', 'F0'): 77 ознак
  ('CIC', 'F0_other_cti'): 83 ознак
  ('CIC', 'F0_service'): 78 ознак
  ('CIC', 'F1'): 84 ознак
  ('UNSW', 'F0'): 41 ознак
  ('UNSW', 'F0_other_cti'): 45 ознак
  ('UNSW', 'F0_service'): 42 ознак
  ('UNSW', 'F1'): 46 ознак
✅ Порядок ознак (не лише кількість) у всіх 8 FEATURE_SETS збігається з незалежним FROZEN_REFERENCE_FEATURES


## 2. Свіже завантаження, frozen train/val/test split, categorical vocab fit-on-train

Категоріальні колонки -- **explicit frozen contract**, а не dtype-inference: той самий урок,
що й з порядком ознак (UNSW, Step 5D.2b) -- автоматичне визначення через
`pd.api.types.is_numeric_dtype(...)` могло б мовчки змінити набір категоріальних колонок,
якби колись змінився dtype джерела (напр. якщо `cve_ctx_service` колись прийде як числовий
код замість рядка). Тому `FROZEN_CATEGORICAL` нижче прописаний буквально (той самий набір,
що й раніше давало dtype-inference в Step 5E.2: CIC=`protocol`,`cve_ctx_service`;
UNSW=`state`,`proto`,`service`,`cve_ctx_service`), з assert-ом, що кожна з них справді є
у `F1`-наборі ознак.

In [3]:
FROZEN_CATEGORICAL = {
    "CIC": ["protocol", "cve_ctx_service"],
    "UNSW": ["state", "proto", "service", "cve_ctx_service"],
}
for dataset_key, cols in FROZEN_CATEGORICAL.items():
    assert all(c in FEATURE_SETS[(dataset_key, "F1")] for c in cols), \
        f"{dataset_key}: FROZEN_CATEGORICAL містить колонку поза FEATURE_SETS F1"
print("✅ FROZEN_CATEGORICAL -- explicit contract, не dtype-inference:", FROZEN_CATEGORICAL)

raw = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = pd.read_parquet(ENRICHED_DIR / cfg["file"])
    raw[dataset_key] = df

splits = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = raw[dataset_key].copy()
    df["_is_attack"] = (df[cfg["label_col"]] != cfg["benign_value"]).astype(int)

    if dataset_key == "UNSW" and "ct_ftp_cmd" in df.columns:
        df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

    t = pd.to_datetime(df[cfg["time_col"]], unit="s") if dataset_key == "UNSW" and pd.api.types.is_integer_dtype(df[cfg["time_col"]]) \
        else pd.to_datetime(df[cfg["time_col"]])

    train_mask = t < cfg["boundary_1"]
    val_mask = (t >= cfg["boundary_1"]) & (t < cfg["boundary_2"])
    test_mask = t >= cfg["boundary_2"]
    train_df, val_df, test_df = df[train_mask].copy(), df[val_mask].copy(), df[test_mask].copy()

    categorical_candidates = FROZEN_CATEGORICAL[dataset_key]
    for col in categorical_candidates:
        train_categories = pd.Categorical(train_df[col]).categories
        train_df[col] = pd.Categorical(train_df[col], categories=train_categories)
        val_df[col] = pd.Categorical(val_df[col], categories=train_categories)
        test_df[col] = pd.Categorical(test_df[col], categories=train_categories)

    splits[dataset_key] = dict(train=train_df, val=val_df, test=test_df, categorical_candidates=categorical_candidates)
    n_train, n_pos = len(train_df), int(train_df["_is_attack"].sum())
    print(f"{dataset_key}: train n={n_train:,}  positive={n_pos:,}  categorical={categorical_candidates}")


✅ FROZEN_CATEGORICAL -- explicit contract, не dtype-inference: {'CIC': ['protocol', 'cve_ctx_service'], 'UNSW': ['state', 'proto', 'service', 'cve_ctx_service']}
CIC: train n=1,979,346  positive=265,848  categorical=['protocol', 'cve_ctx_service']
UNSW: train n=1,441,568  positive=49,346  categorical=['state', 'proto', 'service', 'cve_ctx_service']


## 3. Frozen прогін для кожного з 4 ablation-наборів на датасет (8 моделей всього)

Той самий `train_frozen()`, що й у Step 5E.2 -- ідентичні гіперпараметри, ідентичний seed,
ідентичний train/val/test split, list-equality assert порядку колонок проти frozen
feature-набору.

In [4]:
def train_frozen(dataset_key, feature_set_name):
    features = FEATURE_SETS[(dataset_key, feature_set_name)]
    train_df, val_df, test_df = splits[dataset_key]["train"], splits[dataset_key]["val"], splits[dataset_key]["test"]

    X_train, y_train = train_df[features], train_df["_is_attack"]
    X_val, y_val = val_df[features], val_df["_is_attack"]

    assert list(X_train.columns) == features, f"{dataset_key}-{feature_set_name}: X_train порядок розійшовся з frozen features!"
    assert list(X_val.columns) == features, f"{dataset_key}-{feature_set_name}: X_val порядок розійшовся з frozen features!"

    cat_cols = [c for c in features if X_train[c].dtype.name == "category"]
    n_pos_train = int(y_train.sum())
    n_neg_train = len(y_train) - n_pos_train
    w = n_neg_train / n_pos_train
    params = dict(LGB_PARAMS, scale_pos_weight=w)

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols if cat_cols else "auto")

    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    return model, features


ABLATION_SET_NAMES = ["F0", "F0_service", "F0_other_cti", "F1"]
models = {}
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ABLATION_SET_NAMES:
        model, features = train_frozen(dataset_key, feature_set_name)
        models[(dataset_key, feature_set_name)] = dict(model=model, features=features)
        print(f"{dataset_key}-{feature_set_name}: best_iteration={model.best_iteration}  n_features={len(features)}")


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.105472 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14089
[LightGBM] [Info] Number of data points in the train set: 1979346, number of used features: 69
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134311 -> initscore=-1.863367
[LightGBM] [Info] Start training from score -1.863367
CIC-F0: best_iteration=4  n_features=77
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing 

## 4. Reproducibility audit для F0/F1 -- підтвердження, що той самий frozen протокол
відтворює Step 5D.2 і в ЦЬОМУ notebook (перед тим, як довіряти новим F0_service/F0_other_cti)

Як і в Step 5E.2: перевіряємо мінімум `best_iteration` + `test_ap` + `test_roc_auc` разом
проти frozen референсу (не лише `best_iteration` -- цього недостатньо, прецедент UNSW
Step 5D.2b). `F0_service`/`F0_other_cti` -- нові ablation-набори, для яких frozen референсу
не існує (вони тут вперше), тому для них немає що звіряти -- лише F0/F1 підтверджують, що
довкілля/дані/протокол не розійшлися.

In [5]:
FROZEN_REFERENCE = {
    ("CIC", "F0"): dict(best_iteration=4, test_ap=0.710369, test_roc_auc=0.740553),
    ("CIC", "F1"): dict(best_iteration=118, test_ap=0.974028, test_roc_auc=0.982162),
    ("UNSW", "F0"): dict(best_iteration=33, test_ap=0.973652, test_roc_auc=0.997671),
    ("UNSW", "F1"): dict(best_iteration=290, test_ap=0.983969, test_roc_auc=0.998247),
}

repro_rows = []
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        payload = models[(dataset_key, feature_set_name)]
        model, features = payload["model"], payload["features"]
        val_df, test_df = splits[dataset_key]["val"], splits[dataset_key]["test"]
        y_val, y_test = val_df["_is_attack"].to_numpy(), test_df["_is_attack"].to_numpy()

        p_val = model.predict(val_df[features], num_iteration=model.best_iteration)
        p_test = model.predict(test_df[features], num_iteration=model.best_iteration)

        test_ap = average_precision_score(y_test, p_test)
        test_roc = roc_auc_score(y_test, p_test)

        ref = FROZEN_REFERENCE[(dataset_key, feature_set_name)]
        repro_rows.append(dict(
            dataset=dataset_key, feature_set=feature_set_name,
            best_iteration=int(model.best_iteration), ref_best_iteration=ref["best_iteration"],
            test_ap=test_ap, ref_test_ap=ref["test_ap"],
            test_roc_auc=test_roc, ref_test_roc_auc=ref["test_roc_auc"],
            best_iteration_match=int(model.best_iteration) == ref["best_iteration"],
            test_ap_match=bool(np.isclose(test_ap, ref["test_ap"], rtol=0, atol=1e-6)),
            test_roc_match=bool(np.isclose(test_roc, ref["test_roc_auc"], rtol=0, atol=1e-6)),
        ))

repro_df = pd.DataFrame(repro_rows)
print(repro_df.to_string(index=False))

assert repro_df["best_iteration_match"].all(), "best_iteration НЕ збігається з frozen референсом -- reproduction FAILED"
assert repro_df["test_ap_match"].all(), "test_ap НЕ збігається з frozen референсом -- reproduction FAILED"
assert repro_df["test_roc_match"].all(), "test_roc_auc НЕ збігається з frozen референсом -- reproduction FAILED"

repro_df.to_csv(AUDIT_OUT_DIR / "reproducibility_audit_f0_f1.csv", index=False)
print("\n✅ F0/F1 у цьому notebook відтворюють Step 5D.2 -- той самий frozen протокол; "
      "можна довіряти F0_service/F0_other_cti, навченим за тим самим протоколом нижче.")


dataset feature_set  best_iteration  ref_best_iteration  test_ap  ref_test_ap  test_roc_auc  ref_test_roc_auc  best_iteration_match  test_ap_match  test_roc_match
    CIC          F0               4                   4 0.710369     0.710369      0.740553          0.740553                  True           True            True
    CIC          F1             118                 118 0.974028     0.974028      0.982162          0.982162                  True           True            True
   UNSW          F0              33                  33 0.973652     0.973652      0.997671          0.997671                  True           True            True
   UNSW          F1             290                 290 0.983969     0.983969      0.998247          0.998247                  True           True            True

✅ F0/F1 у цьому notebook відтворюють Step 5D.2 -- той самий frozen протокол; можна довіряти F0_service/F0_other_cti, навченим за тим самим протоколом нижче.


## 5. Метрики для всіх 4 ablation-наборів (val і test AP/ROC-AUC)

In [6]:
ablation_rows = []
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ABLATION_SET_NAMES:
        payload = models[(dataset_key, feature_set_name)]
        model, features = payload["model"], payload["features"]
        val_df, test_df = splits[dataset_key]["val"], splits[dataset_key]["test"]
        y_val, y_test = val_df["_is_attack"].to_numpy(), test_df["_is_attack"].to_numpy()

        p_val = model.predict(val_df[features], num_iteration=model.best_iteration)
        p_test = model.predict(test_df[features], num_iteration=model.best_iteration)

        ablation_rows.append(dict(
            dataset=dataset_key, feature_set=feature_set_name, n_features=len(features),
            best_iteration=int(model.best_iteration),
            val_ap=average_precision_score(y_val, p_val), val_roc_auc=roc_auc_score(y_val, p_val),
            test_ap=average_precision_score(y_test, p_test), test_roc_auc=roc_auc_score(y_test, p_test),
        ))

ablation_df = pd.DataFrame(ablation_rows)
print(ablation_df.to_string(index=False))


dataset  feature_set  n_features  best_iteration   val_ap  val_roc_auc  test_ap  test_roc_auc
    CIC           F0          77               4 0.778550     0.823532 0.710369      0.740553
    CIC   F0_service          78             131 0.649086     0.729430 0.956979      0.978910
    CIC F0_other_cti          83              85 0.652778     0.724039 0.981802      0.987433
    CIC           F1          84             118 0.676896     0.765135 0.974028      0.982162
   UNSW           F0          41              33 0.964633     0.997017 0.973652      0.997671
   UNSW   F0_service          42             288 0.979777     0.997956 0.983679      0.998193
   UNSW F0_other_cti          45             248 0.979036     0.998051 0.983314      0.998417
   UNSW           F1          46             290 0.979776     0.997882 0.983969      0.998247


## 6. ΔAP decomposition: `cve_ctx_service` окремо vs решта CTI окремо vs повний CTI

$$\Delta AP_{service} = AP(F0\_service) - AP(F0)$$
$$\Delta AP_{other} = AP(F0\_other\_cti) - AP(F0)$$
$$\Delta AP_{all} = AP(F1) - AP(F0)$$

Додатково виводимо `sum_check = ΔAP_service + ΔAP_other - ΔAP_all` як ДІАГНОСТИКУ (не assert)
-- нагадування, що LightGBM не є лінійно-адитивною моделлю за ознаками, тому точна рівність
не очікується; сама величина розбіжності показує, наскільки сильно `cve_ctx_service` і
решта CTI взаємодіють (a не діють незалежно) у кожному датасеті.

`service_share_of_all = ΔAP_service / ΔAP_all` -- корисна діагностична величина, але
формулювання при інтерпретації має бути обережним: значення, наприклад, 0.95 читається як
"додавання `cve_ctx_service` окремо відтворює приблизно 95% повного приросту AP у межах
цього frozen experiment", а НЕ як "`cve_ctx_service` причинно відповідає за 95% приросту".

In [7]:
delta_rows = []
for dataset_key in ["CIC", "UNSW"]:
    sub = ablation_df[ablation_df["dataset"] == dataset_key].set_index("feature_set")
    ap_f0 = sub.loc["F0", "test_ap"]
    ap_service = sub.loc["F0_service", "test_ap"]
    ap_other = sub.loc["F0_other_cti", "test_ap"]
    ap_f1 = sub.loc["F1", "test_ap"]

    delta_service = ap_service - ap_f0
    delta_other = ap_other - ap_f0
    delta_all = ap_f1 - ap_f0
    sum_check = (delta_service + delta_other) - delta_all

    delta_rows.append(dict(
        dataset=dataset_key,
        ap_F0=ap_f0, ap_F0_service=ap_service, ap_F0_other_cti=ap_other, ap_F1=ap_f1,
        delta_ap_service=delta_service, delta_ap_other=delta_other, delta_ap_all=delta_all,
        sum_check_service_plus_other_minus_all=sum_check,
        service_share_of_all=(delta_service / delta_all if delta_all != 0 else np.nan),
    ))

delta_df = pd.DataFrame(delta_rows)
print(delta_df.to_string(index=False))
print()
print("Нагадування: sum_check != 0 означає нелінійну взаємодію CTI-ознак у бустингу, "
      "не помилку обчислення.")


dataset    ap_F0  ap_F0_service  ap_F0_other_cti    ap_F1  delta_ap_service  delta_ap_other  delta_ap_all  sum_check_service_plus_other_minus_all  service_share_of_all
    CIC 0.710369       0.956979         0.981802 0.974028          0.246610        0.271434      0.263659                                0.254385              0.935338
   UNSW 0.973652       0.983679         0.983314 0.983969          0.010027        0.009662      0.010317                                0.009372              0.971879

Нагадування: sum_check != 0 означає нелінійну взаємодію CTI-ознак у бустингу, не помилку обчислення.


## 7. Збереження артефактів

In [8]:
ablation_df.to_csv(AUDIT_OUT_DIR / "ablation_metrics_all_sets.csv", index=False)
delta_df.to_csv(AUDIT_OUT_DIR / "delta_ap_decomposition.csv", index=False)

summary = dict(
    note=("Targeted ablation (Phase 3): один-єдиний frozen прогін на набір ознак (seed=42), "
          "без bootstrap/повторних seed-ів -- немає CI. ΔAP-decomposition -- діагностика "
          "величин внеску, не статистичний тест значущості. LightGBM не лінійно-адитивний за "
          "ознаками, тому точна адитивність service+other=all НЕ очікується апріорі."),
    ablation_set_names=ABLATION_SET_NAMES,
    ablation_metrics=ablation_df.to_dict(orient="records"),
    delta_ap_decomposition=delta_df.to_dict(orient="records"),
)
with open(AUDIT_OUT_DIR / "step5e3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("*")):
    print(f"  {p.name}")


Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5e3_targeted_ablation:
  ablation_metrics_all_sets.csv
  delta_ap_decomposition.csv
  reproducibility_audit_f0_f1.csv
  step5e3_summary.json


## Підсумок Phase 3

Заповнюється фактичним виводом після запуску на сервері. Reproducibility audit (розділ 4)
підтверджує -- або спростовує -- що F0/F1 у цьому notebook відтворюють frozen референс
Step 5D.2; лише за умови підтвердження має сенс довіряти ablation-результатам нижче (F0/F1
навчені за тим самим кодом і протоколом, що й F0_service/F0_other_cti).

Ключове порівняння (розділ 6): чи $\Delta AP_{service} \approx \Delta AP_{all}$ для CIC
(тобто приріст AP, отриманий при додаванні `cve_ctx_service` до F0, наближається до приросту
повного CTI-набору; це свідчить про домінування `cve_ctx_service` серед CTI-сигналів у межах
цього frozen experiment -- а НЕ про те, що `cve_ctx_service` "пояснює" чи причинно спричиняє
приріст AP) і чи $\Delta AP_{service} \approx 0$ для UNSW (тобто CTI -- включно з
`cve_ctx_service` -- практично не додає AP понад те, що вже дає сильний native сигнал
`sttl`). Це саме та гіпотеза, яку Step 5E.1 (overlap) і Step 5E.2 (attribution) підготували,
але не довели.

**Формулювання для статті (методологічно коректне):** "Targeted single-feature ablation
under the frozen training protocol shows that adding `cve_ctx_service` alone to F0 accounts
for ΔAP_service = <...> in CIC-IDS2017 (out of ΔAP_all = 0.263659 for the full CTI set),
compared to ΔAP_service = <...> in UNSW-NB15 (out of ΔAP_all = 0.010317). This provides direct
evidence, under this single frozen training run, of the incremental contribution of
`cve_ctx_service` versus the remaining CTI features -- but, absent repeated-seed variance
estimates, these are point estimates, not statistically tested causal claims."

Це завершує заздалегідь визначений (frozen) 3-фазовий дизайн (overlap audit -> feature
attribution -> targeted ablation) для дослідницького питання про асиметрію приросту AP між
CIC-IDS2017 і UNSW-NB15.